In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo mặc định của Pandas cho gọn terminal

def advanced_clean_and_report(file_path, table_name, expected_cols, pk_col):
    print(f"\n{'='*60}")
    print(f"BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: {table_name}")
    print(f"{'='*60}")
    
    if not os.path.exists(file_path):
        print(f"[LỖI] Không tìm thấy file {file_path}")
        return None

    # Đọc dữ liệu
    df = pd.read_csv(file_path)
    
    # Chỉ lấy các cột có trong schema (nếu có trong file thô)
    valid_cols = [col for col in expected_cols if col in df.columns]
    df = df[valid_cols].copy()

    # --- KHỞI TẠO BIẾN THỐNG KÊ ---
    stats = {
        "initial_rows": len(df),
        "initial_cols": len(df.columns),
        "dtypes": df.dtypes.to_dict(),
        "100_percent_missing": [],
        "partial_missing": [],
        "anomalies": [],
        "rounded_cols": [],
        "changes": {
            "to_null_all": 0,
            "median": 0,
            "mean": 0,
            "mode": 0,
            "rounded": 0,
            "deleted_rows": 0,
            "pk_generated": 0 # Thêm biến đếm số PK được tạo mới
        },
        "col_reports": [], 
        "modified_rows": set() 
    }

    # ==========================================
    # 1. XỬ LÝ RIÊNG CHO KHÓA CHÍNH (PK)
    # ==========================================
    if pk_col in df.columns:
        missing_pk_mask = df[pk_col].isnull()
        missing_pk_count = missing_pk_mask.sum()

        if missing_pk_count > 0:
            # Lấy các giá trị PK hiện có, thử ép sang kiểu số (bỏ qua lỗi chữ nếu có)
            existing_pks = pd.to_numeric(df[pk_col], errors='coerce').dropna()
            
            if not existing_pks.empty:
                max_id = int(existing_pks.max())
            else:
                max_id = 0 # Nếu cột rỗng thì bắt đầu từ 0
            
            # Tạo list ID mới (Max + 1, Max + 2,...)
            new_ids = [max_id + i + 1 for i in range(missing_pk_count)]
            
            # Đổ dữ liệu ID mới vào các dòng bị thiếu
            df.loc[missing_pk_mask, pk_col] = new_ids
            
            stats["changes"]["pk_generated"] += missing_pk_count
            stats["col_reports"].append((pk_col, missing_pk_count, "Tự động tạo ID (Max + 1)", missing_pk_count, 0, f"Bắt đầu từ ID: {new_ids[0]}"))

    # ==========================================
    # 2 & 3. XỬ LÝ GIÁ TRỊ THIẾU CÁC CỘT CÒN LẠI
    # ==========================================
    for col in df.columns:
        # Bỏ qua cột PK vì đã xử lý ở trên
        if col == pk_col:
            continue
            
        missing_count = df[col].isnull().sum()
        total_count = len(df)
        
        if missing_count == 0:
            continue
            
        # Kịch bản 1: Thiếu 100%
        if missing_count == total_count:
            df[col] = np.nan 
            stats["100_percent_missing"].append(col)
            stats["changes"]["to_null_all"] += missing_count
            stats["col_reports"].append((col, missing_count, "Chuyển thành NaN (100% thiếu)", missing_count, 0, "Không xóa cột"))
            continue

        # Kịch bản 2: Thiếu 1 phần
        stats["partial_missing"].append(col)
        col_type = df[col].dtype
        method_used = ""
        replaced_val = None
        
        if pd.api.types.is_numeric_dtype(col_type):
            skewness = df[col].skew()
            if pd.isna(skewness): 
                skewness = 0
                
            if abs(skewness) > 1: 
                replaced_val = df[col].median()
                df[col] = df[col].fillna(replaced_val)
                method_used = f"Median (Do dữ liệu bị lệch, skew={skewness:.2f})"
                stats["changes"]["median"] += missing_count
            else: 
                replaced_val = df[col].mean()
                df[col] = df[col].fillna(replaced_val)
                method_used = f"Mean (Dữ liệu phân phối ổn định)"
                stats["changes"]["mean"] += missing_count
        else:
            modes = df[col].mode()
            if not modes.empty:
                replaced_val = modes[0]
                df[col] = df[col].fillna(replaced_val)
                method_used = "Mode (Giá trị xuất hiện nhiều nhất)"
                stats["changes"]["mode"] += missing_count
            else:
                method_used = "Giữ nguyên (Không tìm được mode)"

        if replaced_val is not None:
            stats["col_reports"].append((col, missing_count, method_used, missing_count, 0, f"Thay bằng: {replaced_val}"))

    # ==========================================
    # 4. LÀM TRÒN CÁC CỘT SỐ
    # ==========================================
    for col in df.columns:
        if col == pk_col: # Thường PK không cần làm tròn
            continue
            
        if pd.api.types.is_float_dtype(df[col]):
            rounded_series = df[col].round(2)
            changed_mask = df[col] != rounded_series
            changed_count = changed_mask.sum()
            
            if changed_count > 0:
                df[col] = rounded_series
                stats["rounded_cols"].append((col, 2))
                stats["changes"]["rounded"] += changed_count
                stats["col_reports"].append((col, "-", "Làm tròn số", 0, changed_count, "Làm tròn 2 chữ số thập phân"))

    # ==========================================
    # 5. PHÁT HIỆN DỮ LIỆU KHÔNG ĐỒNG NHẤT
    # ==========================================
    for col in df.columns:
        if df[col].dtype == object:
            temp_numeric = pd.to_numeric(df[col], errors='coerce')
            if temp_numeric.notna().sum() > 0 and temp_numeric.notna().sum() < len(df[col].dropna()):
                anomaly_mask = temp_numeric.isna() & df[col].notna()
                anomaly_indices = df[anomaly_mask].index.tolist()
                anomaly_values = df.loc[anomaly_mask, col].unique()
                
                if anomaly_indices:
                    stats["anomalies"].append({
                        "col": col,
                        "expected": "Numeric/Mixed",
                        "values": anomaly_values[:5], 
                        "indices": anomaly_indices[:10], 
                        "count": len(anomaly_indices)
                    })

    # ==========================================
    # OUTPUT BÁO CÁO THEO YÊU CẦU
    # ==========================================
    
    print("\n[1] TỔNG QUAN DỮ LIỆU")
    print(f" - Số dòng ban đầu: {stats['initial_rows']}")
    print(f" - Số cột: {stats['initial_cols']}")
    
    print("\n[2] GIÁ TRỊ THIẾU")
    print(f" - Các cột 100% missing: {stats['100_percent_missing'] if stats['100_percent_missing'] else 'Không có'}")
    print(f" - Các cột missing một phần: {stats['partial_missing'] if stats['partial_missing'] else 'Không có'}")
    
    print("\n[3] DỮ LIỆU KHÔNG ĐỒNG NHẤT (CẢNH BÁO)")
    if not stats["anomalies"]:
        print(" - Không phát hiện bất thường nghiêm trọng.")
    else:
        for anomaly in stats["anomalies"]:
            print(f" - Cột: {anomaly['col']}")
            print(f"   + Số lượng record bất thường: {anomaly['count']}")
            print(f"   + Giá trị bất thường (mẫu): {anomaly['values']}")

    print("\n[4] LÀM TRÒN DỮ LIỆU")
    if not stats["rounded_cols"]:
        print(" - Không có cột nào cần làm tròn.")
    else:
        for r_col in stats["rounded_cols"]:
            print(f" - Cột '{r_col[0]}' được làm tròn đến {r_col[1]} chữ số thập phân.")

    print("\n[5] THỐNG KÊ THAY ĐỔI")
    print(f" - Số Khóa chính (PK) được tự tạo mới: {stats['changes']['pk_generated']}")
    print(f" - Số giá trị chuyển thành null/NaN: {stats['changes']['to_null_all']}")
    print(f" - Số giá trị missing thay bằng Median: {stats['changes']['median']}")
    print(f" - Số giá trị missing thay bằng Mean: {stats['changes']['mean']}")
    print(f" - Số giá trị missing thay bằng Mode: {stats['changes']['mode']}")
    print(f" - Số giá trị được làm tròn: {stats['changes']['rounded']}")
    
    print("\n[CHI TIẾT THAY ĐỔI TỪNG CỘT]")
    if stats["col_reports"]:
        report_df = pd.DataFrame(stats["col_reports"], columns=["Cột", "Missing ban đầu", "Phương pháp", "Số giá trị thay", "Số giá trị làm tròn", "Ghi chú"])
        print(report_df.to_string(index=False))
    else:
        print(" - Không có thay đổi nào.")

    # Xuất file
    output_filename = f"{table_name}_SILVER.csv"
    df.to_csv(output_filename, index=False)
    print(f"\n=> Đã lưu file làm sạch thành công: {output_filename}")
    
    return df

# ==========================================
# THỰC THI CHO GEOGRAPHY VÀ PRODUCTS
# ==========================================
if __name__ == "__main__":
    SCHEMA_CONFIG = [
        {
            "file_raw": "GEOGRAPHY.csv",
            "table_name": "GEOGRAPHY",
            "columns": ["zip", "city", "district", "region"],
            "pk": "zip" # Bổ sung định nghĩa PK ở đây
        },
        {
            "file_raw": "PRODUCTS_FINAL.csv",
            "table_name": "PRODUCTS",
            "columns": ["product_id", "product_name", "category", "segment", "size", "color", "price", "cogs"],
            "pk": "product_id" # Bổ sung định nghĩa PK ở đây
        }
    ]

    for config in SCHEMA_CONFIG:
        advanced_clean_and_report(
            file_path=config["file_raw"],
            table_name=config["table_name"],
            expected_cols=config["columns"],
            pk_col=config["pk"] # Truyền PK vào hàm
        )


BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: GEOGRAPHY

[1] TỔNG QUAN DỮ LIỆU
 - Số dòng ban đầu: 39948
 - Số cột: 4

[2] GIÁ TRỊ THIẾU
 - Các cột 100% missing: Không có
 - Các cột missing một phần: Không có

[3] DỮ LIỆU KHÔNG ĐỒNG NHẤT (CẢNH BÁO)
 - Không phát hiện bất thường nghiêm trọng.

[4] LÀM TRÒN DỮ LIỆU
 - Không có cột nào cần làm tròn.

[5] THỐNG KÊ THAY ĐỔI
 - Số Khóa chính (PK) được tự tạo mới: 0
 - Số giá trị chuyển thành null/NaN: 0
 - Số giá trị missing thay bằng Median: 0
 - Số giá trị missing thay bằng Mean: 0
 - Số giá trị missing thay bằng Mode: 0
 - Số giá trị được làm tròn: 0

[CHI TIẾT THAY ĐỔI TỪNG CỘT]
 - Không có thay đổi nào.

=> Đã lưu file làm sạch thành công: GEOGRAPHY_CLEANED.csv

BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: PRODUCTS

[1] TỔNG QUAN DỮ LIỆU
 - Số dòng ban đầu: 2412
 - Số cột: 8

[2] GIÁ TRỊ THIẾU
 - Các cột 100% missing: Không có
 - Các cột missing một phần: Không có

[3] DỮ LIỆU KHÔNG ĐỒNG NHẤT (CẢNH BÁO)
 - Không phát hiện bất thường nghiêm trọng.

[4] LÀ

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo mặc định của Pandas

def advanced_clean_and_report(file_path, table_name, expected_cols, pk_col, fk_configs=None):
    print(f"\n{'='*70}")
    print(f"BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: {table_name}")
    print(f"{'='*70}")
    
    if not os.path.exists(file_path):
        print(f"[LỖI] Không tìm thấy file {file_path}")
        return None

    # Đọc dữ liệu
    df = pd.read_csv(file_path)
    
    # Chỉ lấy các cột có trong schema
    valid_cols = [col for col in expected_cols if col in df.columns]
    missing_cols = [col for col in expected_cols if col not in df.columns]
    if missing_cols:
        print(f"[*] Cảnh báo: File thô đang thiếu các cột sau: {missing_cols}")
        
    df = df[valid_cols].copy()

    # Xóa dòng rỗng hoàn toàn và duplicate hoàn toàn
    initial_len = len(df)
    df = df.dropna(how='all').drop_duplicates()
    deleted_garbage = initial_len - len(df)

    # --- KHỞI TẠO BIẾN THỐNG KÊ ---
    stats = {
        "initial_rows": initial_len,
        "initial_cols": len(df.columns),
        "100_percent_missing": [],
        "partial_missing": [],
        "anomalies": [],
        "rounded_cols": [],
        "fk_reports": [],
        "changes": {
            "to_null_all": 0,
            "median": 0,
            "mean": 0,
            "mode": 0,
            "rounded": 0,
            "pk_generated": 0,
            "deleted_garbage": deleted_garbage
        },
        "col_reports": []
    }

    # ==========================================
    # 1. XỬ LÝ RIÊNG CHO KHÓA CHÍNH (PK)
    # ==========================================
    if pk_col in df.columns:
        missing_pk_mask = df[pk_col].isnull()
        missing_pk_count = missing_pk_mask.sum()

        if missing_pk_count > 0:
            existing_pks = pd.to_numeric(df[pk_col], errors='coerce').dropna()
            max_id = int(existing_pks.max()) if not existing_pks.empty else 0
            
            new_ids = [max_id + i + 1 for i in range(missing_pk_count)]
            df.loc[missing_pk_mask, pk_col] = new_ids
            
            stats["changes"]["pk_generated"] += missing_pk_count
            stats["col_reports"].append((pk_col, missing_pk_count, "Tự tạo ID (Max + 1)", missing_pk_count, 0, f"Bắt đầu từ: {new_ids[0]}"))

    # ==========================================
    # 2. KIỂM TRA KHÓA NGOẠI (FK) - CHỈ BÁO CÁO, KHÔNG XÓA
    # ==========================================
    if fk_configs:
        for fk_conf in fk_configs:
            fk_col = fk_conf['fk_col']
            parent_file = fk_conf['parent_file']
            parent_pk = fk_conf['parent_pk']
            
            if fk_col in df.columns:
                if not os.path.exists(parent_file):
                    stats["fk_reports"].append(f"[LỖI FK] Thiếu file {parent_file} để đối chiếu cột {fk_col}.")
                    continue
                    
                df_parent = pd.read_csv(parent_file)
                if parent_pk not in df_parent.columns:
                    stats["fk_reports"].append(f"[LỖI FK] Cột {parent_pk} không có trong {parent_file}.")
                    continue
                    
                child_fks = df[fk_col].dropna().unique()
                parent_pks = df_parent[parent_pk].dropna().unique()
                orphan_fks = set(child_fks) - set(parent_pks)
                
                if orphan_fks:
                    orphan_records = df[df[fk_col].isin(orphan_fks)]
                    stats["fk_reports"].append({
                        "fk_col": fk_col,
                        "parent_file": parent_file,
                        "orphan_count": len(orphan_fks),
                        "orphan_values": list(orphan_fks)[:5],
                        "record_count": len(orphan_records)
                    })

    # ==========================================
    # 3. XỬ LÝ GIÁ TRỊ THIẾU CÁC CỘT CÒN LẠI
    # ==========================================
    for col in df.columns:
        if col == pk_col:
            continue
            
        missing_count = df[col].isnull().sum()
        if missing_count == 0:
            continue
            
        if missing_count == len(df):
            df[col] = np.nan 
            stats["100_percent_missing"].append(col)
            stats["changes"]["to_null_all"] += missing_count
            stats["col_reports"].append((col, missing_count, "Chuyển thành NaN (100% thiếu)", missing_count, 0, "Không xóa cột"))
            continue

        stats["partial_missing"].append(col)
        col_type = df[col].dtype
        replaced_val = None
        
        if pd.api.types.is_numeric_dtype(col_type):
            skewness = df[col].skew() if not pd.isna(df[col].skew()) else 0
            if abs(skewness) > 1: 
                replaced_val = df[col].median()
                method_used = f"Median (Skew={skewness:.2f})"
                stats["changes"]["median"] += missing_count
            else: 
                replaced_val = df[col].mean()
                method_used = f"Mean (Phân phối ổn)"
                stats["changes"]["mean"] += missing_count
        else:
            modes = df[col].mode()
            if not modes.empty:
                replaced_val = modes[0]
                method_used = "Mode"
                stats["changes"]["mode"] += missing_count
            else:
                method_used = "Giữ nguyên"

        if replaced_val is not None:
            df[col] = df[col].fillna(replaced_val)
            stats["col_reports"].append((col, missing_count, method_used, missing_count, 0, f"Thay bằng: {replaced_val}"))

    # ==========================================
    # 4. LÀM TRÒN SỐ & PHÁT HIỆN BẤT THƯỜNG
    # ==========================================
    for col in df.columns:
        if col == pk_col:
            continue
            
        # Làm tròn
        if pd.api.types.is_float_dtype(df[col]):
            rounded_series = df[col].round(2)
            changed_count = (df[col] != rounded_series).sum()
            if changed_count > 0:
                df[col] = rounded_series
                stats["rounded_cols"].append(col)
                stats["changes"]["rounded"] += changed_count
                stats["col_reports"].append((col, "-", "Làm tròn số", 0, changed_count, "Làm tròn 2 số thập phân"))

        # Tìm dòng bị trộn chữ vào số (anomalies)
        if df[col].dtype == object:
            temp_numeric = pd.to_numeric(df[col], errors='coerce')
            if temp_numeric.notna().sum() > 0 and temp_numeric.notna().sum() < len(df[col].dropna()):
                anomaly_mask = temp_numeric.isna() & df[col].notna()
                if anomaly_mask.sum() > 0:
                    stats["anomalies"].append({
                        "col": col,
                        "count": anomaly_mask.sum(),
                        "values": df.loc[anomaly_mask, col].unique()[:5]
                    })

    # ==========================================
    # OUTPUT BÁO CÁO 
    # ==========================================
    print("\n[1] TỔNG QUAN")
    print(f" - Số dòng ban đầu: {stats['initial_rows']}")
    print(f" - Số dòng rác/trùng lặp đã xóa: {stats['changes']['deleted_garbage']}")
    
    print("\n[2] BÁO CÁO KHÓA NGOẠI (FK)")
    if not fk_configs:
        print(" - Không thiết lập kiểm tra Khóa ngoại.")
    elif not stats["fk_reports"]:
        print(" -> Hợp lệ 100%. Tất cả FK đều khớp với bảng cha.")
    else:
        for report in stats["fk_reports"]:
            if isinstance(report, str):
                print(f" {report}")
            else:
                print(f" [CẢNH BÁO] Cột '{report['fk_col']}' có {report['orphan_count']} giá trị không có trong {report['parent_file']}")
                print(f"    + Số dòng bị ảnh hưởng: {report['record_count']}")
                print(f"    + Vài giá trị mồ côi mẫu: {report['orphan_values']}")

    print("\n[3] GIÁ TRỊ THIẾU & LÀM TRÒN")
    print(f" - Cột missing 100%: {stats['100_percent_missing'] if stats['100_percent_missing'] else 'Không'}")
    print(f" - Cột missing 1 phần: {stats['partial_missing'] if stats['partial_missing'] else 'Không'}")
    print(f" - Số PK tự tạo (Max+1): {stats['changes']['pk_generated']}")
    
    if stats["anomalies"]:
        print("\n[4] DỮ LIỆU KHÔNG ĐỒNG NHẤT (Mixed Type)")
        for anomaly in stats["anomalies"]:
            print(f" - Cột '{anomaly['col']}': {anomaly['count']} dòng bất thường. Mẫu: {anomaly['values']}")

    print("\n[CHI TIẾT THAY ĐỔI TỪNG CỘT]")
    if stats["col_reports"]:
        report_df = pd.DataFrame(stats["col_reports"], columns=["Cột", "Missing gốc", "Phương pháp", "Đã thay", "Đã làm tròn", "Ghi chú"])
        print(report_df.to_string(index=False))
    else:
        print(" - Không có thay đổi nào.")

    output_filename = f"{table_name}_SILVER.csv"
    df.to_csv(output_filename, index=False)
    print(f"\n=> Đã lưu file thành công: {output_filename}")
    
    return df


if __name__ == "__main__":
    # Thiết lập config cho CUSTOMERS
    CUSTOMERS_CONFIG = {
        "file_raw": "customers.csv",
        "table_name": "CUSTOMERS",
        "columns": ["customer_id", "gender", "age_group", "zip", "acquisition_channel", "signup_date"],
        "pk": "customer_id",
        # List các khóa ngoại cần check. Truyền tên file cha đã làm sạch vô đây.
        "fks": [
            {
                "fk_col": "zip", 
                "parent_file": "GEOGRAPHY_SILVER.csv", 
                "parent_pk": "zip"
            }
        ]
    }

    advanced_clean_and_report(
        file_path=CUSTOMERS_CONFIG["file_raw"],
        table_name=CUSTOMERS_CONFIG["table_name"],
        expected_cols=CUSTOMERS_CONFIG["columns"],
        pk_col=CUSTOMERS_CONFIG["pk"],
        fk_configs=CUSTOMERS_CONFIG["fks"]
    )


BẮT ĐẦU PHÂN TÍCH VÀ LÀM SẠCH BẢNG: CUSTOMERS

[1] TỔNG QUAN
 - Số dòng ban đầu: 121930
 - Số dòng rác/trùng lặp đã xóa: 0

[2] BÁO CÁO KHÓA NGOẠI (FK)
 -> Hợp lệ 100%. Tất cả FK đều khớp với bảng cha.

[3] GIÁ TRỊ THIẾU & LÀM TRÒN
 - Cột missing 100%: Không
 - Cột missing 1 phần: Không
 - Số PK tự tạo (Max+1): 0

[CHI TIẾT THAY ĐỔI TỪNG CỘT]
 - Không có thay đổi nào.

=> Đã lưu file thành công: CUSTOMERS_CLEANED.csv
